In [ ]:
# Cell 1 — Load config
%run /home/jovyan/work/setup/config.py
import sys; sys.path.insert(0, "/home/jovyan/work")
from utils.delta_utils import save_layer

In [ ]:
# Cell 2 — Build dim_channel (merge result from Requirement 1)
# This dimension is the product of the CSV merge — it contains data from both source files.
from pyspark.sql.functions import xxhash64, col

df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/silver_beverage_sales_enriched")

dim_channel = (
    df_silver
    .select("trade_chnl_desc", "chnl_group", "trade_group_desc", "trade_type_desc")
    .dropDuplicates(["trade_chnl_desc"])
    .withColumn("channel_sk", xxhash64(col("trade_chnl_desc")))
    .select("channel_sk", "trade_chnl_desc", "chnl_group", "trade_group_desc", "trade_type_desc")
)

save_layer(dim_channel, "dim_channel", GOLD_PATH, PG_WRITE_PROPS)
dim_channel.show(30, truncate=False)